In [ ]:
import pyterrier as pt
if not pt.started():
  pt.init()

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/workspace/corpus.csv')

In [ ]:
df

In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/content/drive/MyDrive/workspace/topics.xml')
data = tree.getroot()
titles = []
title_ids = []
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    title_ids.append(num)
topics = pd.DataFrame(titles,columns = ['query'])
topics['qid'] = topics.index + 51
columns_titles = ["qid","query"]
topics=topics.reindex(columns=columns_titles)

In [ ]:
qrels_path=("/content/drive/MyDrive/workspace/qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

In [ ]:
docnos = qrels.docno
# list(docnos)

In [ ]:
df_test = df.copy()
df_test = df_test[df_test['docno'].isin(docnos)]
df_test

In [ ]:
pd_indexer = pt.DFIndexer("./pd_index")
indexref = pd_indexer.index(df_test["text"], df_test["docno"])

In [ ]:
index = pt.IndexFactory.of(indexref)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res=retr.transform(topics)

In [ ]:
res

In [ ]:
eval = pt.Utils.evaluate(res,qrels,metrics=["ndcg_cut_5", "bpref"])
eval

In [ ]:
pd_indexer = pt.DFIndexer("./pd_index_full")
indexref = pd_indexer.index(df["text"], df["docno"])

In [ ]:
index = pt.IndexFactory.of(indexref)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res=retr.transform(topics)

In [ ]:
eval = pt.Utils.evaluate(res,qrels,metrics=["ndcg_cut_5", "bpref"])
eval